<a href="https://colab.research.google.com/github/Manawariqbal/LLM-FIneTuning/blob/main/Simple_Fine_Tuning_of_a_Small_Causal_LM_(DistilGPT2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install if needed
# !pip install transformers datasets accelerate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset

In [2]:
# 1. Load a small pre-trained generative model (transfer learning starting point)
model_name = "distilbert/distilgpt2"  # ~82M params, fast to experiment with
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Important for causal LM

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [10]:
# 2. Load a small dataset
!pip install --upgrade datasets huggingface_hub

from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:1%]")  # tiny slice for demo

# Filter out empty text lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Filter out any samples that mapped to 0 tokens to avoid empty tensors during training
tokenized_dataset = tokenized_dataset.filter(lambda x: len(x["input_ids"]) > 0)

Filter:   0%|          | 0/258 [00:00<?, ? examples/s]

In [11]:
# 3. Data collator for causal language modeling (next-token prediction)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [12]:
# 4. Training arguments (fine-tuning hyperparameters)
training_args = TrainingArguments(
    output_dir="./distilgpt2-finetuned",
    per_device_train_batch_size=4,
    num_train_epochs=1,          # Keep small for demo
    learning_rate=5e-5,
    logging_steps=50,
    save_steps=500,
    fp16=True,                   # Use mixed precision if GPU available
    report_to="none"
)

In [13]:
# 5. Trainer (handles the actual fine-tuning)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [14]:
# 6. Fine-tune!
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,3.998281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=65, training_loss=4.005801685039813, metrics={'train_runtime': 349.1658, 'train_samples_per_second': 0.739, 'train_steps_per_second': 0.186, 'total_flos': 6822805192704.0, 'train_loss': 4.005801685039813, 'epoch': 1.0})

In [15]:
# 7. Save and test generation
model.save_pretrained("./distilgpt2-finetuned")
tokenizer.save_pretrained("./distilgpt2-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilgpt2-finetuned/tokenizer_config.json',
 './distilgpt2-finetuned/tokenizer.json')

In [17]:
# Quick generation test
prompt = "Transfer learning in generative AI"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.8)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Transfer learning in generative AI systems that were developed by the early 1990s.






































































